# 🧠 Advanced DSA in Python — Complete Cheat Sheet

> **Level:** Advanced | **Language:** Python 3.10+  
> Covers theory, patterns, complexity analysis, and battle-tested implementations.

## 📚 Table of Contents
1. [Complexity Analysis](#1)
2. [Arrays & Strings](#2)
3. [Two Pointers & Sliding Window](#3)
4. [Linked Lists](#4)
5. [Stacks & Queues](#5)
6. [Recursion & Backtracking](#6)
7. [Binary Search](#7)
8. [Trees — BST, AVL, Segment Tree, Fenwick Tree](#8)
9. [Heaps & Priority Queues](#9)
10. [Tries (Prefix Trees)](#10)
11. [Graphs — BFS, DFS, Topological Sort, SCC](#11)
12. [Shortest Paths — Dijkstra, Bellman-Ford, Floyd-Warshall](#12)
13. [Minimum Spanning Tree — Kruskal, Prim](#13)
14. [Union-Find (Disjoint Set Union)](#14)
15. [Sorting Algorithms](#15)
16. [Dynamic Programming Patterns](#16)
17. [Greedy Algorithms](#17)
18. [Bit Manipulation](#18)
19. [Math & Number Theory](#19)
20. [Advanced Patterns Cheatsheet](#20)

---
## 1. Complexity Analysis <a id='1'></a>

### Big-O Hierarchy (fastest → slowest)
```
O(1) < O(log n) < O(√n) < O(n) < O(n log n) < O(n²) < O(n³) < O(2ⁿ) < O(n!)
```

### Rules
- **Drop constants:** `O(2n)` → `O(n)`
- **Drop lower terms:** `O(n² + n)` → `O(n²)`
- **Different inputs → different variables:** `O(a + b)` not `O(n)`
- **Space complexity** counts stack frames too (recursion!)

### Recurrence Relations (Master Theorem)
For `T(n) = aT(n/b) + f(n)`:
- `f(n) = O(n^(log_b(a) - ε))` → `T(n) = Θ(n^log_b(a))`
- `f(n) = Θ(n^log_b(a))` → `T(n) = Θ(n^log_b(a) · log n)`
- `f(n) = Ω(n^(log_b(a) + ε))` → `T(n) = Θ(f(n))`

**Examples:**
- Merge Sort: `T(n) = 2T(n/2) + O(n)` → `O(n log n)` (Case 2)
- Binary Search: `T(n) = T(n/2) + O(1)` → `O(log n)` (Case 2)

In [ ]:
import time, sys, math, heapq, collections, functools, itertools
from typing import Optional, List, Dict, Tuple, Any
from collections import defaultdict, deque, Counter, OrderedDict
from functools import lru_cache, cache
import bisect

# Utility: measure runtime
def timeit(fn, *args, **kwargs):
    start = time.perf_counter()
    result = fn(*args, **kwargs)
    end = time.perf_counter()
    print(f"{fn.__name__}: {(end-start)*1000:.4f} ms")
    return result

print("Imports ready ✅")

---
## 2. Arrays & Strings <a id='2'></a>

### Key Concepts
- Python `list` is a **dynamic array** (amortized O(1) append)
- **Prefix sums** reduce range-query from O(n) to O(1)
- **Kadane's algorithm** solves max-subarray in O(n)
- **Moore's Voting** finds majority element in O(n) time, O(1) space
- **Dutch National Flag** sorts 3-valued arrays in O(n)

### Prefix Sum Pattern
```
pre[i] = arr[0] + arr[1] + ... + arr[i-1]
sum(l, r) = pre[r+1] - pre[l]   ← O(1) range sum
```

In [ ]:
# ─── PREFIX SUM ──────────────────────────────────────────────────
def build_prefix(arr):
    pre = [0] * (len(arr) + 1)
    for i, v in enumerate(arr):
        pre[i+1] = pre[i] + v
    return pre

def range_sum(pre, l, r):   # inclusive [l, r]
    return pre[r+1] - pre[l]

arr = [3, 1, 4, 1, 5, 9, 2, 6]
pre = build_prefix(arr)
print(f"Sum [2,5] = {range_sum(pre, 2, 5)}")   # 4+1+5+9 = 19

# ─── 2D PREFIX SUM ───────────────────────────────────────────────
def build_prefix_2d(matrix):
    m, n = len(matrix), len(matrix[0])
    pre = [[0]*(n+1) for _ in range(m+1)]
    for i in range(m):
        for j in range(n):
            pre[i+1][j+1] = (matrix[i][j] + pre[i][j+1]
                             + pre[i+1][j] - pre[i][j])
    return pre

def region_sum(pre, r1, c1, r2, c2):  # inclusive rectangle
    return (pre[r2+1][c2+1] - pre[r1][c2+1]
            - pre[r2+1][c1] + pre[r1][c1])

matrix = [[1,2,3],[4,5,6],[7,8,9]]
p2 = build_prefix_2d(matrix)
print(f"2D region sum [0,0→1,1] = {region_sum(p2, 0, 0, 1, 1)}")  # 12

In [ ]:
# ─── KADANE'S ALGORITHM — Maximum Subarray ───────────────────────
def max_subarray(nums):
    max_sum = cur = nums[0]
    start = end = s = 0
    for i in range(1, len(nums)):
        if cur < 0:
            cur = nums[i]; s = i
        else:
            cur += nums[i]
        if cur > max_sum:
            max_sum = cur; start, end = s, i
    return max_sum, start, end

nums = [-2, 1, -3, 4, -1, 2, 1, -5, 4]
print(f"Max subarray: {max_subarray(nums)}")  # 6, subarray [4,-1,2,1]

# ─── MOORE'S VOTING — Majority Element ───────────────────────────
def majority_element(nums):   # element appearing > n/2 times
    candidate, count = None, 0
    for n in nums:
        if count == 0: candidate = n
        count += 1 if n == candidate else -1
    return candidate

print(majority_element([3,2,3]))       # 3
print(majority_element([2,2,1,1,1,2,2]))  # 2

# ─── DUTCH NATIONAL FLAG ─────────────────────────────────────────
def sort_colors(nums):  # in-place, 0s, 1s, 2s
    lo = mid = 0; hi = len(nums) - 1
    while mid <= hi:
        if nums[mid] == 0:
            nums[lo], nums[mid] = nums[mid], nums[lo]; lo += 1; mid += 1
        elif nums[mid] == 1:
            mid += 1
        else:
            nums[mid], nums[hi] = nums[hi], nums[mid]; hi -= 1
    return nums

print(sort_colors([2,0,2,1,1,0]))  # [0,0,1,1,2,2]

In [ ]:
# ─── STRING ALGORITHMS ───────────────────────────────────────────

# KMP — Pattern Matching O(n+m)
def kmp_search(text, pattern):
    """Returns all start indices of pattern in text."""
    def build_lps(p):
        lps = [0] * len(p)
        length = 0; i = 1
        while i < len(p):
            if p[i] == p[length]:
                length += 1; lps[i] = length; i += 1
            elif length:
                length = lps[length - 1]
            else:
                lps[i] = 0; i += 1
        return lps

    lps = build_lps(pattern)
    results, i, j = [], 0, 0
    while i < len(text):
        if text[i] == pattern[j]:
            i += 1; j += 1
        if j == len(pattern):
            results.append(i - j); j = lps[j-1]
        elif i < len(text) and text[i] != pattern[j]:
            j = lps[j-1] if j else (i := i+1, 0)[1]
    return results

print(kmp_search("aabxaabxaaab", "aabx"))  # [0, 4]

# Rabin-Karp — Rolling Hash O(n+m) avg
def rabin_karp(text, pattern):
    BASE, MOD = 31, 10**9 + 9
    n, m = len(text), len(pattern)
    if m > n: return []
    pw = pow(BASE, m-1, MOD)
    def h(s): return sum((ord(c)-96)*pow(BASE,i,MOD) for i,c in enumerate(s)) % MOD
    ph = h(pattern); wh = h(text[:m])
    results = []
    if ph == wh and text[:m] == pattern: results.append(0)
    for i in range(1, n-m+1):
        wh = (wh - (ord(text[i-1])-96)*pw) * BASE % MOD + (ord(text[i+m-1])-96)
        wh %= MOD
        if wh == ph and text[i:i+m] == pattern: results.append(i)
    return results

print(rabin_karp("aabxaabxaaab", "aabx"))  # [0, 4]

# Longest Palindromic Substring — Manacher's O(n)
def manacher(s):
    t = '#' + '#'.join(s) + '#'
    n = len(t); p = [0]*n
    c = r = 0
    for i in range(n):
        mirror = 2*c - i
        if i < r: p[i] = min(r - i, p[mirror])
        while i+p[i]+1 < n and i-p[i]-1 >= 0 and t[i+p[i]+1] == t[i-p[i]-1]:
            p[i] += 1
        if i + p[i] > r: c, r = i, i + p[i]
    max_len, center = max((v, i) for i, v in enumerate(p))
    start = (center - max_len) // 2
    return s[start:start+max_len]

print(manacher("babad"))    # bab
print(manacher("racecar"))  # racecar

---
## 3. Two Pointers & Sliding Window <a id='3'></a>

### When to use
| Pattern | Trigger | Complexity |
|---|---|---|
| Two Pointers (opposite ends) | Sorted array, pair/triplet sum | O(n) |
| Two Pointers (same direction) | Merge, partition, fast-slow | O(n) |
| Fixed Window | Subarray of size k | O(n) |
| Variable Window | Longest/shortest subarray with condition | O(n) |

### Variable Window Template
```python
left = 0; state = {}; ans = 0
for right, val in enumerate(arr):
    # expand: add arr[right] to state
    while <invalid>:     # shrink window
        # remove arr[left] from state
        left += 1
    ans = max(ans, right - left + 1)
```

In [ ]:
# ─── TWO POINTERS ────────────────────────────────────────────────

def three_sum(nums):       # All unique triplets summing to 0
    nums.sort(); res = []
    for i in range(len(nums)-2):
        if i > 0 and nums[i] == nums[i-1]: continue
        l, r = i+1, len(nums)-1
        while l < r:
            s = nums[i]+nums[l]+nums[r]
            if s == 0:
                res.append([nums[i],nums[l],nums[r]])
                while l<r and nums[l]==nums[l+1]: l+=1
                while l<r and nums[r]==nums[r-1]: r-=1
                l+=1; r-=1
            elif s < 0: l += 1
            else: r -= 1
    return res

print(three_sum([-1,0,1,2,-1,-4]))  # [[-1,-1,2],[-1,0,1]]

# Trapping Rain Water — O(n) time, O(1) space
def trap(height):
    l, r = 0, len(height)-1
    lmax = rmax = water = 0
    while l < r:
        if height[l] < height[r]:
            if height[l] >= lmax: lmax = height[l]
            else: water += lmax - height[l]
            l += 1
        else:
            if height[r] >= rmax: rmax = height[r]
            else: water += rmax - height[r]
            r -= 1
    return water

print(trap([0,1,0,2,1,0,1,3,2,1,2,1]))  # 6

# Container With Most Water
def max_area(height):
    l, r, mx = 0, len(height)-1, 0
    while l < r:
        mx = max(mx, min(height[l],height[r])*(r-l))
        if height[l] < height[r]: l += 1
        else: r -= 1
    return mx

print(max_area([1,8,6,2,5,4,8,3,7]))  # 49

In [ ]:
# ─── SLIDING WINDOW ──────────────────────────────────────────────

# Longest substring without repeating characters
def length_of_longest_substring(s):
    seen = {}; l = ans = 0
    for r, c in enumerate(s):
        if c in seen and seen[c] >= l:
            l = seen[c] + 1
        seen[c] = r
        ans = max(ans, r - l + 1)
    return ans

print(length_of_longest_substring("abcabcbb"))   # 3
print(length_of_longest_substring("pwwkew"))     # 3

# Minimum window substring
def min_window(s, t):
    need = Counter(t); missing = len(t)
    best = ""; l = 0
    for r, c in enumerate(s):
        if need[c] > 0: missing -= 1
        need[c] -= 1
        if missing == 0:
            while need[s[l]] < 0: need[s[l]] += 1; l += 1
            if not best or r-l+1 < len(best): best = s[l:r+1]
            need[s[l]] += 1; missing += 1; l += 1
    return best

print(min_window("ADOBECODEBANC", "ABC"))  # BANC

# Max sum of subarray of size k (fixed window)
def max_sum_k(arr, k):
    window = sum(arr[:k]); best = window
    for i in range(k, len(arr)):
        window += arr[i] - arr[i-k]
        best = max(best, window)
    return best

print(max_sum_k([2,3,4,1,5], 3))  # 10 (3+4+1? No... 4+1+5=10) ✓

# Sliding window maximum — Monotonic deque O(n)
def sliding_window_max(nums, k):
    dq = deque()  # stores indices, front = max
    res = []
    for i, n in enumerate(nums):
        while dq and nums[dq[-1]] <= n: dq.pop()
        dq.append(i)
        if dq[0] == i - k: dq.popleft()
        if i >= k-1: res.append(nums[dq[0]])
    return res

print(sliding_window_max([1,3,-1,-3,5,3,6,7], 3))  # [3,3,5,5,6,7]

---
## 4. Linked Lists <a id='4'></a>

### Key Techniques
- **Fast/slow pointers** → cycle detection, middle node, kth from end
- **Dummy head** → simplifies edge cases on insertion/deletion
- **Reverse in-place** → O(n) time, O(1) space
- **Floyd's Cycle Detection:** fast meets slow at distance `μ` from head to cycle start

In [ ]:
class ListNode:
    def __init__(self, val=0, nxt=None):
        self.val = val; self.next = nxt
    def __repr__(self):
        vals = []; cur = self
        while cur: vals.append(str(cur.val)); cur = cur.next
        return ' -> '.join(vals)

def make_list(arr):
    dummy = ListNode(0); cur = dummy
    for v in arr: cur.next = ListNode(v); cur = cur.next
    return dummy.next

# ─── REVERSE ─────────────────────────────────────────────────────
def reverse_list(head):
    prev = None
    while head:
        nxt = head.next; head.next = prev; prev = head; head = nxt
    return prev

def reverse_between(head, left, right):  # Reverse sublist [left..right]
    dummy = ListNode(0, head); pre = dummy
    for _ in range(left-1): pre = pre.next
    cur = pre.next
    for _ in range(right-left):
        nxt = cur.next; cur.next = nxt.next
        nxt.next = pre.next; pre.next = nxt
    return dummy.next

lst = make_list([1,2,3,4,5])
print(reverse_list(lst))                     # 5->4->3->2->1
lst = make_list([1,2,3,4,5])
print(reverse_between(lst, 2, 4))            # 1->4->3->2->5

# ─── FLOYD'S CYCLE ───────────────────────────────────────────────
def detect_cycle(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next; fast = fast.next.next
        if slow is fast:                     # cycle found
            slow = head
            while slow is not fast:
                slow = slow.next; fast = fast.next
            return slow                      # cycle start node
    return None

# ─── MERGE SORT on LINKED LIST — O(n log n) ──────────────────────
def sort_list(head):
    if not head or not head.next: return head
    # Find middle
    slow, fast = head, head.next
    while fast and fast.next:
        slow = slow.next; fast = fast.next.next
    mid = slow.next; slow.next = None
    # Merge
    def merge(l1, l2):
        dummy = cur = ListNode()
        while l1 and l2:
            if l1.val <= l2.val: cur.next = l1; l1 = l1.next
            else: cur.next = l2; l2 = l2.next
            cur = cur.next
        cur.next = l1 or l2
        return dummy.next
    return merge(sort_list(head), sort_list(mid))

print(sort_list(make_list([4,2,1,3])))   # 1->2->3->4

# ─── REORDER LIST  L0→L1→…→Ln-1→Ln  →  L0→Ln→L1→Ln-1→… ────────
def reorder_list(head):
    slow = fast = head
    while fast.next and fast.next.next:
        slow = slow.next; fast = fast.next.next
    second = reverse_list(slow.next); slow.next = None
    first = head
    while second:
        tmp1, tmp2 = first.next, second.next
        first.next = second; second.next = tmp1
        first = tmp1; second = tmp2

lst = make_list([1,2,3,4,5]); reorder_list(lst); print(lst)  # 1->5->2->4->3

---
## 5. Stacks & Queues <a id='5'></a>

### Monotonic Stack — Key Pattern
- **Monotonic Increasing Stack** → tracks "previous smaller element"
- **Monotonic Decreasing Stack** → tracks "previous larger element"
- Used for: Next Greater Element, Largest Rectangle, Stock Span

### Deque (collections.deque)
- O(1) append/pop from both ends
- Use as **sliding window max/min** (monotonic deque)

In [ ]:
# ─── MONOTONIC STACK ─────────────────────────────────────────────

def next_greater_element(nums):
    """For each element, find the next greater element to its right."""
    n = len(nums); res = [-1]*n; stack = []
    for i, v in enumerate(nums):
        while stack and nums[stack[-1]] < v:
            res[stack.pop()] = v
        stack.append(i)
    return res

print(next_greater_element([2,1,2,4,3]))  # [4,2,4,-1,-1]

def largest_rectangle_histogram(heights):
    """O(n) using monotonic increasing stack."""
    stack = []; max_area = 0
    heights = heights + [0]  # sentinel
    for i, h in enumerate(heights):
        start = i
        while stack and stack[-1][1] > h:
            idx, ht = stack.pop()
            max_area = max(max_area, ht * (i - idx))
            start = idx
        stack.append((start, h))
    return max_area

print(largest_rectangle_histogram([2,1,5,6,2,3]))  # 10

def daily_temperatures(temperatures):
    """Days until warmer. Classic monotonic stack."""
    res = [0]*len(temperatures); stack = []
    for i, t in enumerate(temperatures):
        while stack and temperatures[stack[-1]] < t:
            j = stack.pop(); res[j] = i - j
        stack.append(i)
    return res

print(daily_temperatures([73,74,75,71,69,72,76,73]))  # [1,1,4,2,1,1,0,0]

# ─── MIN STACK — O(1) getMin ─────────────────────────────────────
class MinStack:
    def __init__(self):
        self.stack = []; self.min_stack = []
    def push(self, val):
        self.stack.append(val)
        m = min(val, self.min_stack[-1] if self.min_stack else val)
        self.min_stack.append(m)
    def pop(self): self.stack.pop(); self.min_stack.pop()
    def top(self): return self.stack[-1]
    def get_min(self): return self.min_stack[-1]

ms = MinStack(); [ms.push(x) for x in [3,1,2]]
print(ms.get_min(), ms.top())  # 1, 2

# ─── LRU CACHE — O(1) get/put ────────────────────────────────────
class LRUCache:
    def __init__(self, capacity):
        self.cap = capacity
        self.cache = OrderedDict()  # key -> value, ordered by usage
    def get(self, key):
        if key not in self.cache: return -1
        self.cache.move_to_end(key)
        return self.cache[key]
    def put(self, key, value):
        if key in self.cache: self.cache.move_to_end(key)
        self.cache[key] = value
        if len(self.cache) > self.cap: self.cache.popitem(last=False)

lru = LRUCache(2)
lru.put(1,1); lru.put(2,2); print(lru.get(1))  # 1
lru.put(3,3); print(lru.get(2))                # -1 (evicted)

---
## 6. Recursion & Backtracking <a id='6'></a>

### Backtracking Template
```python
def backtrack(state, choices):
    if is_solution(state):
        results.append(state[:]); return
    for choice in choices:
        if is_valid(choice, state):
            state.append(choice)       # choose
            backtrack(state, next_choices)
            state.pop()                # un-choose
```

### Pruning Strategies
- Skip duplicates (sort + check previous)
- Early termination (sum too large, depth exceeded)
- Bitmask visited set instead of array

In [ ]:
# ─── PERMUTATIONS ────────────────────────────────────────────────
def permutations(nums):
    res = []
    def bt(path, used):
        if len(path) == len(nums): res.append(path[:]); return
        for i, n in enumerate(nums):
            if used[i]: continue
            if i > 0 and nums[i]==nums[i-1] and not used[i-1]: continue  # skip dups
            used[i] = True; path.append(n)
            bt(path, used)
            path.pop(); used[i] = False
    nums.sort(); bt([], [False]*len(nums))
    return res

print(len(permutations([1,1,2])), "unique permutations")  # 3

# ─── COMBINATIONS SUM ────────────────────────────────────────────
def combination_sum(candidates, target):
    candidates.sort(); res = []
    def bt(start, path, rem):
        if rem == 0: res.append(path[:]); return
        for i in range(start, len(candidates)):
            if candidates[i] > rem: break  # pruning!
            path.append(candidates[i])
            bt(i, path, rem - candidates[i])  # reuse same element
            path.pop()
    bt(0, [], target); return res

print(combination_sum([2,3,6,7], 7))  # [[2,2,3],[7]]

# ─── N-QUEENS ────────────────────────────────────────────────────
def n_queens(n):
    res = []; cols = set(); diag1 = set(); diag2 = set()
    board = [['.']*n for _ in range(n)]
    def bt(row):
        if row == n:
            res.append([''.join(r) for r in board]); return
        for col in range(n):
            if col in cols or (row-col) in diag1 or (row+col) in diag2: continue
            board[row][col] = 'Q'
            cols.add(col); diag1.add(row-col); diag2.add(row+col)
            bt(row+1)
            board[row][col] = '.'
            cols.discard(col); diag1.discard(row-col); diag2.discard(row+col)
    bt(0); return res

print(f"N-Queens(4): {len(n_queens(4))} solutions")  # 2

# ─── WORD SEARCH ─────────────────────────────────────────────────
def word_search(board, word):
    m, n = len(board), len(board[0])
    def dfs(i, j, k):
        if k == len(word): return True
        if not(0<=i<m and 0<=j<n) or board[i][j] != word[k]: return False
        tmp, board[i][j] = board[i][j], '#'  # mark visited
        found = any(dfs(i+di,j+dj,k+1) for di,dj in [(0,1),(0,-1),(1,0),(-1,0)])
        board[i][j] = tmp                    # restore
        return found
    return any(dfs(i,j,0) for i in range(m) for j in range(n))

board = [["A","B","C","E"],["S","F","C","S"],["A","D","E","E"]]
print(word_search(board, "ABCCED"))  # True

---
## 7. Binary Search <a id='7'></a>

### Template — "Find first True" (left boundary)
```python
lo, hi = 0, n
while lo < hi:
    mid = (lo + hi) // 2
    if condition(mid): hi = mid   # could be answer
    else: lo = mid + 1
return lo
```

### Key Insight
Binary search works on **any monotonic predicate**, not just sorted arrays.
- "Is it possible to do X with capacity C?" → binary search on C!

In [ ]:
# ─── CLASSIC VARIANTS ────────────────────────────────────────────
def lower_bound(arr, target):   # First index >= target
    return bisect.bisect_left(arr, target)

def upper_bound(arr, target):   # First index > target
    return bisect.bisect_right(arr, target)

# Search in Rotated Sorted Array
def search_rotated(nums, target):
    lo, hi = 0, len(nums)-1
    while lo <= hi:
        mid = (lo+hi)//2
        if nums[mid] == target: return mid
        if nums[lo] <= nums[mid]:          # left half sorted
            if nums[lo] <= target < nums[mid]: hi = mid-1
            else: lo = mid+1
        else:                              # right half sorted
            if nums[mid] < target <= nums[hi]: lo = mid+1
            else: hi = mid-1
    return -1

print(search_rotated([4,5,6,7,0,1,2], 0))  # 4
print(search_rotated([4,5,6,7,0,1,2], 3))  # -1

# Find Peak Element
def find_peak(nums):
    lo, hi = 0, len(nums)-1
    while lo < hi:
        mid = (lo+hi)//2
        if nums[mid] > nums[mid+1]: hi = mid
        else: lo = mid+1
    return lo

print(find_peak([1,2,3,1]))    # 2
print(find_peak([1,2,1,3,5,6,4]))  # 5

# ─── BINARY SEARCH ON ANSWER ─────────────────────────────────────

# Minimum capacity to ship packages within D days
def ship_within_days(weights, days):
    def feasible(cap):
        d, cur = 1, 0
        for w in weights:
            if w > cap: return False
            if cur + w > cap: d += 1; cur = 0
            cur += w
        return d <= days
    lo, hi = max(weights), sum(weights)
    while lo < hi:
        mid = (lo+hi)//2
        if feasible(mid): hi = mid
        else: lo = mid+1
    return lo

print(ship_within_days([1,2,3,4,5,6,7,8,9,10], 5))  # 15

# Koko eating bananas
def min_eating_speed(piles, h):
    lo, hi = 1, max(piles)
    while lo < hi:
        mid = (lo+hi)//2
        if sum(math.ceil(p/mid) for p in piles) <= h: hi = mid
        else: lo = mid+1
    return lo

print(min_eating_speed([3,6,7,11], 8))  # 4

---
## 8. Trees <a id='8'></a>

### Tree Traversal Summary
| Traversal | Order | Iterative Tool |
|---|---|---|
| Inorder | Left → Root → Right | Stack |
| Preorder | Root → Left → Right | Stack |
| Postorder | Left → Right → Root | Two-Stack / Reverse |
| Level Order | BFS level by level | Queue |
| Morris Traversal | Inorder, O(1) space | Thread pointers |

### Segment Tree
- Point update + Range query in **O(log n)**
- Lazy propagation for **range update + range query**

### Fenwick Tree (BIT)
- Prefix sums with updates in **O(log n)** — simpler than segment tree

In [ ]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val; self.left = left; self.right = right

def make_tree(vals):  # BFS construction from list
    if not vals: return None
    root = TreeNode(vals[0]); q = deque([root]); i = 1
    while q and i < len(vals):
        node = q.popleft()
        if i < len(vals) and vals[i] is not None:
            node.left = TreeNode(vals[i]); q.append(node.left)
        i += 1
        if i < len(vals) and vals[i] is not None:
            node.right = TreeNode(vals[i]); q.append(node.right)
        i += 1
    return root

# ─── TRAVERSALS ──────────────────────────────────────────────────
def inorder_iterative(root):
    res, stack, cur = [], [], root
    while cur or stack:
        while cur: stack.append(cur); cur = cur.left
        cur = stack.pop(); res.append(cur.val); cur = cur.right
    return res

def level_order(root):
    if not root: return []
    q, res = deque([root]), []
    while q:
        level = []
        for _ in range(len(q)):
            node = q.popleft(); level.append(node.val)
            if node.left: q.append(node.left)
            if node.right: q.append(node.right)
        res.append(level)
    return res

# Morris Inorder — O(n) time, O(1) space
def morris_inorder(root):
    res, cur = [], root
    while cur:
        if not cur.left:
            res.append(cur.val); cur = cur.right
        else:
            pred = cur.left
            while pred.right and pred.right is not cur:
                pred = pred.right
            if not pred.right:
                pred.right = cur; cur = cur.left
            else:
                pred.right = None; res.append(cur.val); cur = cur.right
    return res

tree = make_tree([1,2,3,4,5,None,7])
print("Inorder:", inorder_iterative(tree))   # [4,2,5,1,3,7]
print("Morris: ", morris_inorder(tree))      # [4,2,5,1,3,7]
print("Levels: ", level_order(tree))         # [[1],[2,3],[4,5,7]]

In [ ]:
# ─── BST OPERATIONS ──────────────────────────────────────────────

def lca_bst(root, p, q):  # Lowest Common Ancestor in BST — O(h)
    while root:
        if p < root.val and q < root.val: root = root.left
        elif p > root.val and q > root.val: root = root.right
        else: return root.val

def lca_binary_tree(root, p, q):  # LCA in general Binary Tree
    if not root or root.val in (p, q): return root
    left = lca_binary_tree(root.left, p, q)
    right = lca_binary_tree(root.right, p, q)
    return root if left and right else left or right

def diameter(root):  # Longest path between any two nodes
    ans = [0]
    def height(node):
        if not node: return 0
        l, r = height(node.left), height(node.right)
        ans[0] = max(ans[0], l + r)
        return 1 + max(l, r)
    height(root); return ans[0]

def max_path_sum(root):  # Max path sum (can start/end anywhere)
    best = [float('-inf')]
    def dfs(node):
        if not node: return 0
        l = max(dfs(node.left), 0)
        r = max(dfs(node.right), 0)
        best[0] = max(best[0], node.val + l + r)
        return node.val + max(l, r)
    dfs(root); return best[0]

tree = make_tree([1,2,3,4,5])
print("Diameter:", diameter(tree))        # 3
print("Max path sum:", max_path_sum(tree)) # 11 (4+2+1+3+5... no. 4+2+5=11? let's see)

In [ ]:
# ─── SEGMENT TREE with LAZY PROPAGATION ─────────────────────────

class SegmentTree:
    """Range Sum Query + Range Update with Lazy Propagation — O(log n)"""
    def __init__(self, nums):
        self.n = len(nums)
        self.tree = [0] * (4 * self.n)
        self.lazy = [0] * (4 * self.n)
        self._build(nums, 1, 0, self.n-1)

    def _build(self, nums, node, start, end):
        if start == end: self.tree[node] = nums[start]; return
        mid = (start+end)//2
        self._build(nums, 2*node,   start, mid)
        self._build(nums, 2*node+1, mid+1, end)
        self.tree[node] = self.tree[2*node] + self.tree[2*node+1]

    def _push_down(self, node, start, end):
        if self.lazy[node]:
            mid = (start+end)//2
            for child, s, e in [(2*node,start,mid),(2*node+1,mid+1,end)]:
                self.tree[child] += self.lazy[node] * (e-s+1)
                self.lazy[child] += self.lazy[node]
            self.lazy[node] = 0

    def update(self, l, r, val, node=1, start=None, end=None):
        if start is None: start, end = 0, self.n-1
        if r < start or end < l: return
        if l <= start and end <= r:
            self.tree[node] += val*(end-start+1); self.lazy[node] += val; return
        self._push_down(node, start, end); mid = (start+end)//2
        self.update(l, r, val, 2*node, start, mid)
        self.update(l, r, val, 2*node+1, mid+1, end)
        self.tree[node] = self.tree[2*node] + self.tree[2*node+1]

    def query(self, l, r, node=1, start=None, end=None):
        if start is None: start, end = 0, self.n-1
        if r < start or end < l: return 0
        if l <= start and end <= r: return self.tree[node]
        self._push_down(node, start, end); mid = (start+end)//2
        return (self.query(l, r, 2*node, start, mid) +
                self.query(l, r, 2*node+1, mid+1, end))

st = SegmentTree([1,2,3,4,5])
print("Sum[0,4] =", st.query(0,4))   # 15
st.update(1, 3, 10)                  # add 10 to indices 1..3
print("Sum[0,4] =", st.query(0,4))   # 15+30 = 45

# ─── FENWICK TREE (BIT) ──────────────────────────────────────────
class FenwickTree:
    """1-indexed. Point update + Prefix sum query — O(log n)."""
    def __init__(self, n): self.n = n; self.bit = [0]*(n+1)
    def update(self, i, delta):   # i is 1-indexed
        while i <= self.n: self.bit[i] += delta; i += i & (-i)
    def query(self, i):           # prefix sum [1..i]
        s = 0
        while i > 0: s += self.bit[i]; i -= i & (-i)
        return s
    def range_query(self, l, r):  # sum [l..r]
        return self.query(r) - self.query(l-1)

ft = FenwickTree(5)
for i, v in enumerate([1,2,3,4,5], 1): ft.update(i, v)
print("BIT prefix[3] =", ft.query(3))      # 6
print("BIT range[2,4] =", ft.range_query(2,4))  # 9

---
## 9. Heaps & Priority Queues <a id='9'></a>

### Python's `heapq` — Min-Heap by default
- For **max-heap**: negate values `-v`
- For **custom key**: push `(priority, item)` tuples
- `heapq.nlargest(k, it)` / `heapq.nsmallest(k, it)` → O(n log k)

### Key Applications
| Problem | Pattern |
|---|---|
| K-th largest/smallest | Min-heap of size k |
| Merge k sorted lists | Min-heap with (val, list_id, idx) |
| Sliding window median | Two heaps (max + min) |
| Dijkstra's shortest path | Min-heap on (dist, node) |

In [ ]:
# ─── KTH LARGEST ─────────────────────────────────────────────────
def kth_largest(nums, k):
    heap = []
    for n in nums:
        heapq.heappush(heap, n)
        if len(heap) > k: heapq.heappop(heap)
    return heap[0]

print(kth_largest([3,2,1,5,6,4], 2))   # 5

# ─── MERGE K SORTED LISTS ────────────────────────────────────────
def merge_k_lists(lists):
    dummy = cur = ListNode(0)
    heap = []
    for i, node in enumerate(lists):
        if node: heapq.heappush(heap, (node.val, i, node))
    while heap:
        val, i, node = heapq.heappop(heap)
        cur.next = node; cur = cur.next
        if node.next: heapq.heappush(heap, (node.next.val, i, node.next))
    return dummy.next

lists = [make_list([1,4,5]), make_list([1,3,4]), make_list([2,6])]
print(merge_k_lists(lists))   # 1->1->2->3->4->4->5->6

# ─── SLIDING WINDOW MEDIAN — Two Heaps ───────────────────────────
import heapq
class MedianFinder:
    """Supports addNum and findMedian in O(log n)."""
    def __init__(self):
        self.small = []   # max-heap (negate)
        self.large = []   # min-heap
    def add_num(self, num):
        heapq.heappush(self.small, -num)
        heapq.heappush(self.large, -heapq.heappop(self.small))
        if len(self.large) > len(self.small):
            heapq.heappush(self.small, -heapq.heappop(self.large))
    def find_median(self):
        if len(self.small) > len(self.large): return -self.small[0]
        return (-self.small[0] + self.large[0]) / 2

mf = MedianFinder()
for n in [1,2,3,4,5]: mf.add_num(n)
print("Median:", mf.find_median())   # 3.0

# ─── TOP K FREQUENT ELEMENTS ─────────────────────────────────────
def top_k_frequent(nums, k):
    count = Counter(nums)
    return heapq.nlargest(k, count.keys(), key=count.get)

print(top_k_frequent([1,1,1,2,2,3], 2))   # [1,2]

---
## 10. Tries (Prefix Trees) <a id='10'></a>

### Applications
- Autocomplete / typeahead
- Word dictionary with wildcard search
- XOR maximum (Binary Trie)
- IP routing (Longest Prefix Match)

In [ ]:
# ─── TRIE — Insert, Search, Prefix, Delete ───────────────────────
class TrieNode:
    def __init__(self):
        self.children = {}
        self.is_end = False
        self.count = 0  # how many words pass through

class Trie:
    def __init__(self): self.root = TrieNode()

    def insert(self, word):
        cur = self.root
        for c in word:
            cur.children.setdefault(c, TrieNode())
            cur = cur.children[c]; cur.count += 1
        cur.is_end = True

    def search(self, word):
        cur = self.root
        for c in word:
            if c not in cur.children: return False
            cur = cur.children[c]
        return cur.is_end

    def starts_with(self, prefix):
        cur = self.root
        for c in prefix:
            if c not in cur.children: return False
            cur = cur.children[c]
        return True

    def count_starts_with(self, prefix):
        cur = self.root
        for c in prefix:
            if c not in cur.children: return 0
            cur = cur.children[c]
        return cur.count

    def search_wildcard(self, word):  # '.' matches any char
        def dfs(node, i):
            if i == len(word): return node.is_end
            c = word[i]
            if c == '.':
                return any(dfs(child, i+1) for child in node.children.values())
            if c not in node.children: return False
            return dfs(node.children[c], i+1)
        return dfs(self.root, 0)

t = Trie()
for w in ["apple","app","apply","banana"]: t.insert(w)
print(t.search("app"))           # True
print(t.starts_with("appl"))     # True
print(t.count_starts_with("app")) # 3
print(t.search_wildcard("a..le")) # True

# ─── BINARY TRIE — Maximum XOR ───────────────────────────────────
class BinaryTrie:
    """Finds pair with maximum XOR in O(n * 32)."""
    def __init__(self): self.root = {}

    def insert(self, num):
        cur = self.root
        for i in range(31, -1, -1):
            bit = (num >> i) & 1
            cur = cur.setdefault(bit, {})

    def max_xor_with(self, num):
        cur = self.root; xor = 0
        for i in range(31, -1, -1):
            bit = (num >> i) & 1
            want = 1 - bit
            if want in cur: cur = cur[want]; xor |= (1 << i)
            elif bit in cur: cur = cur[bit]
        return xor

def find_maximum_xor(nums):
    bt = BinaryTrie()
    for n in nums: bt.insert(n)
    return max(bt.max_xor_with(n) for n in nums)

print(find_maximum_xor([3,10,5,25,2,8]))  # 28 (5 XOR 25)

---
## 11. Graphs <a id='11'></a>

### Representations
- **Adjacency List:** `defaultdict(list)` — O(V+E) space
- **Adjacency Matrix:** `[[0]*V for _ in range(V)]` — O(V²) space
- **Edge List:** `[(u,v,w), ...]`

### Traversal Comparison
| Algorithm | Time | Space | Use Case |
|---|---|---|---|
| BFS | O(V+E) | O(V) | Shortest path (unweighted), level traversal |
| DFS | O(V+E) | O(V) | Cycle detection, topological sort, SCC |
| Dijkstra | O((V+E) log V) | O(V) | Shortest path (non-negative weights) |
| Bellman-Ford | O(VE) | O(V) | Negative weights, detect negative cycles |
| Floyd-Warshall | O(V³) | O(V²) | All-pairs shortest path |

In [ ]:
# ─── BFS & DFS ───────────────────────────────────────────────────

def bfs(graph, start):
    visited, queue, order = {start}, deque([start]), []
    while queue:
        node = queue.popleft(); order.append(node)
        for nei in graph[node]:
            if nei not in visited:
                visited.add(nei); queue.append(nei)
    return order

def dfs(graph, start, visited=None):
    if visited is None: visited = set()
    visited.add(start)
    for nei in graph[start]:
        if nei not in visited: dfs(graph, nei, visited)
    return visited

# ─── TOPOLOGICAL SORT (Kahn's BFS) ──────────────────────────────
def topological_sort_kahn(n, prerequisites):
    """Course Schedule — n courses, prerequisites = [(a,b)] meaning b→a"""
    graph = defaultdict(list); indegree = [0]*n
    for a, b in prerequisites:
        graph[b].append(a); indegree[a] += 1
    q = deque(i for i in range(n) if indegree[i] == 0)
    order = []
    while q:
        node = q.popleft(); order.append(node)
        for nei in graph[node]:
            indegree[nei] -= 1
            if indegree[nei] == 0: q.append(nei)
    return order if len(order) == n else []  # [] means cycle

print(topological_sort_kahn(4, [[1,0],[2,0],[3,1],[3,2]]))  # [0,1,2,3] or [0,2,1,3]

# ─── CYCLE DETECTION — Directed Graph (DFS coloring) ────────────
def has_cycle_directed(n, edges):
    graph = defaultdict(list)
    for u, v in edges: graph[u].append(v)
    WHITE, GRAY, BLACK = 0, 1, 2
    color = [WHITE]*n
    def dfs(u):
        color[u] = GRAY
        for v in graph[u]:
            if color[v] == GRAY: return True   # back edge → cycle
            if color[v] == WHITE and dfs(v): return True
        color[u] = BLACK; return False
    return any(dfs(i) for i in range(n) if color[i] == WHITE)

print(has_cycle_directed(4, [(0,1),(1,2),(2,0)]))  # True
print(has_cycle_directed(4, [(0,1),(1,2),(2,3)]))  # False

# ─── STRONGLY CONNECTED COMPONENTS — Kosaraju's ──────────────────
def kosaraju_scc(n, edges):
    graph = defaultdict(list); rev_graph = defaultdict(list)
    for u, v in edges:
        graph[u].append(v); rev_graph[v].append(u)
    visited = [False]*n; order = []
    def dfs1(u):
        visited[u] = True
        for v in graph[u]:
            if not visited[v]: dfs1(v)
        order.append(u)
    for i in range(n):
        if not visited[i]: dfs1(i)
    visited = [False]*n; sccs = []
    def dfs2(u, scc):
        visited[u] = True; scc.append(u)
        for v in rev_graph[u]:
            if not visited[v]: dfs2(v, scc)
    for u in reversed(order):
        if not visited[u]:
            scc = []; dfs2(u, scc); sccs.append(scc)
    return sccs

print(kosaraju_scc(5, [(0,2),(2,1),(1,0),(0,3),(3,4)]))
# [[4],[3],[0,1,2]] — two SCCs

In [ ]:
# ─── BIPARTITE CHECK ─────────────────────────────────────────────
def is_bipartite(graph):
    n = len(graph); color = [-1]*n
    for start in range(n):
        if color[start] != -1: continue
        q = deque([start]); color[start] = 0
        while q:
            u = q.popleft()
            for v in graph[u]:
                if color[v] == -1: color[v] = 1-color[u]; q.append(v)
                elif color[v] == color[u]: return False
    return True

print(is_bipartite([[1,3],[0,2],[1,3],[0,2]]))  # True

# ─── NUMBER OF ISLANDS — Multi-source BFS ────────────────────────
def num_islands(grid):
    if not grid: return 0
    m, n, count = len(grid), len(grid[0]), 0
    def bfs(i, j):
        q = deque([(i,j)]); grid[i][j] = '0'
        while q:
            r, c = q.popleft()
            for dr, dc in [(0,1),(0,-1),(1,0),(-1,0)]:
                nr, nc = r+dr, c+dc
                if 0<=nr<m and 0<=nc<n and grid[nr][nc]=='1':
                    grid[nr][nc]='0'; q.append((nr,nc))
    for i in range(m):
        for j in range(n):
            if grid[i][j]=='1': bfs(i,j); count+=1
    return count

grid = [["1","1","0"],["1","1","0"],["0","0","1"]]
print(num_islands(grid))   # 2

---
## 12. Shortest Paths <a id='12'></a>

In [ ]:
# ─── DIJKSTRA'S — O((V+E) log V) ────────────────────────────────
def dijkstra(graph, src, n):
    """graph[u] = [(v, weight), ...]"""
    dist = [float('inf')] * n; dist[src] = 0
    heap = [(0, src)]      # (dist, node)
    while heap:
        d, u = heapq.heappop(heap)
        if d > dist[u]: continue   # stale entry
        for v, w in graph[u]:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                heapq.heappush(heap, (dist[v], v))
    return dist

graph = defaultdict(list)
for u,v,w in [(0,1,4),(0,2,1),(2,1,2),(1,3,1),(2,3,5)]:
    graph[u].append((v,w)); graph[v].append((u,w))
print("Dijkstra:", dijkstra(graph, 0, 4))  # [0, 3, 1, 4]

# ─── BELLMAN-FORD — O(VE), handles negative weights ──────────────
def bellman_ford(n, edges, src):
    """edges = [(u, v, weight)]"""
    dist = [float('inf')] * n; dist[src] = 0
    for _ in range(n - 1):
        for u, v, w in edges:
            if dist[u] != float('inf') and dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
    # Check for negative cycles
    for u, v, w in edges:
        if dist[u] != float('inf') and dist[u] + w < dist[v]:
            return None  # negative cycle exists
    return dist

edges = [(0,1,4),(0,2,5),(1,2,-3),(2,3,2)]
print("Bellman-Ford:", bellman_ford(4, edges, 0))

# ─── FLOYD-WARSHALL — O(V³), all-pairs ──────────────────────────
def floyd_warshall(n, edges):
    INF = float('inf')
    dist = [[INF]*n for _ in range(n)]
    for i in range(n): dist[i][i] = 0
    for u, v, w in edges: dist[u][v] = min(dist[u][v], w)
    for k in range(n):
        for i in range(n):
            for j in range(n):
                if dist[i][k] + dist[k][j] < dist[i][j]:
                    dist[i][j] = dist[i][k] + dist[k][j]
    return dist

print("Floyd-Warshall[0][3]:", floyd_warshall(4, edges)[0][3])

---
## 13. Minimum Spanning Tree <a id='13'></a>

In [ ]:
# ─── KRUSKAL'S MST — O(E log E) ──────────────────────────────────
def kruskal(n, edges):
    """edges = [(weight, u, v)]"""
    parent = list(range(n)); rank = [0]*n
    def find(x):
        while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]
        return x
    def union(x, y):
        px, py = find(x), find(y)
        if px == py: return False
        if rank[px] < rank[py]: px, py = py, px
        parent[py] = px
        if rank[px] == rank[py]: rank[px] += 1
        return True
    mst_weight = 0; mst_edges = []
    for w, u, v in sorted(edges):
        if union(u, v):
            mst_weight += w; mst_edges.append((u,v,w))
    return mst_weight, mst_edges

edges = [(1,0,1),(4,0,2),(2,1,2),(5,1,3),(3,2,3)]
weight, mst = kruskal(4, edges)
print(f"MST weight: {weight}, edges: {mst}")

# ─── PRIM'S MST — O(E log V) with min-heap ───────────────────────
def prim(n, graph):
    """graph[u] = [(v, w)]"""
    visited = [False]*n; heap = [(0, 0)]; total = 0
    while heap:
        w, u = heapq.heappop(heap)
        if visited[u]: continue
        visited[u] = True; total += w
        for v, wt in graph[u]:
            if not visited[v]: heapq.heappush(heap, (wt, v))
    return total

g = defaultdict(list)
for w, u, v in edges:
    g[u].append((v,w)); g[v].append((u,w))
print("Prim's MST weight:", prim(4, g))

---
## 14. Union-Find (DSU) <a id='14'></a>

### With Path Compression + Union by Rank
- Nearly **O(1)** per operation (amortized O(α(n)))
- Essential for: Kruskal's, cycle detection, dynamic connectivity

In [ ]:
class DSU:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0]*n
        self.size = [1]*n
        self.components = n

    def find(self, x):   # Path compression
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]

    def union(self, x, y):  # Union by rank
        px, py = self.find(x), self.find(y)
        if px == py: return False
        if self.rank[px] < self.rank[py]: px, py = py, px
        self.parent[py] = px
        self.size[px] += self.size[py]
        if self.rank[px] == self.rank[py]: self.rank[px] += 1
        self.components -= 1
        return True

    def connected(self, x, y): return self.find(x) == self.find(y)
    def component_size(self, x): return self.size[self.find(x)]

# Example: Number of connected components
dsu = DSU(5)
for u, v in [(0,1),(1,2),(3,4)]:
    dsu.union(u, v)
print(f"Components: {dsu.components}")     # 2
print(f"0 and 2 connected: {dsu.connected(0,2)}")  # True
print(f"0 and 3 connected: {dsu.connected(0,3)}")  # False

# Accounts Merge problem pattern
def accounts_merge(accounts):
    email_to_id = {}
    email_to_name = {}
    dsu = DSU(len(accounts) * 10)
    uid = 0
    for name, *emails in accounts:
        for email in emails:
            if email not in email_to_id:
                email_to_id[email] = uid; uid += 1
            email_to_name[email_to_id[emails[0]]] = name
            dsu.union(email_to_id[emails[0]], email_to_id[email])
    groups = defaultdict(list)
    for email, eid in email_to_id.items():
        groups[dsu.find(eid)].append(email)
    return [[email_to_name[k]] + sorted(v) for k, v in groups.items()]

print("DSU demo complete ✅")

---
## 15. Sorting Algorithms <a id='15'></a>

| Algorithm | Best | Avg | Worst | Space | Stable |
|---|---|---|---|---|---|
| Bubble | O(n) | O(n²) | O(n²) | O(1) | ✅ |
| Selection | O(n²) | O(n²) | O(n²) | O(1) | ❌ |
| Insertion | O(n) | O(n²) | O(n²) | O(1) | ✅ |
| Merge | O(n log n) | O(n log n) | O(n log n) | O(n) | ✅ |
| Quick | O(n log n) | O(n log n) | O(n²) | O(log n) | ❌ |
| Heap | O(n log n) | O(n log n) | O(n log n) | O(1) | ❌ |
| Counting | O(n+k) | O(n+k) | O(n+k) | O(k) | ✅ |
| Radix | O(nk) | O(nk) | O(nk) | O(n+k) | ✅ |

In [ ]:
# ─── MERGE SORT ──────────────────────────────────────────────────
def merge_sort(arr):
    if len(arr) <= 1: return arr
    mid = len(arr)//2
    left = merge_sort(arr[:mid]); right = merge_sort(arr[mid:])
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]: result.append(left[i]); i+=1
        else: result.append(right[j]); j+=1
    return result + left[i:] + right[j:]

# Merge sort counting inversions simultaneously
def count_inversions(arr):
    if len(arr) <= 1: return arr, 0
    mid = len(arr)//2
    left, lc = count_inversions(arr[:mid])
    right, rc = count_inversions(arr[mid:])
    merged = []; inv = lc + rc; i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]: merged.append(left[i]); i+=1
        else: merged.append(right[j]); inv += len(left)-i; j+=1
    return merged + left[i:] + right[j:], inv

print(count_inversions([3,1,2,5,4]))  # ([1,2,3,4,5], 3)

# ─── QUICK SORT with 3-way partition ────────────────────────────
def quick_sort(arr, lo=0, hi=None):
    if hi is None: hi = len(arr)-1
    if lo >= hi: return
    # 3-way partition (handles duplicates efficiently)
    pivot = arr[(lo+hi)//2]
    lt = lo; gt = hi; i = lo
    while i <= gt:
        if arr[i] < pivot: arr[lt],arr[i] = arr[i],arr[lt]; lt+=1; i+=1
        elif arr[i] > pivot: arr[i],arr[gt] = arr[gt],arr[i]; gt-=1
        else: i+=1
    quick_sort(arr, lo, lt-1); quick_sort(arr, gt+1, hi)

# ─── HEAP SORT ───────────────────────────────────────────────────
def heap_sort(arr):
    n = len(arr)
    def heapify(arr, n, i):
        largest = i; l, r = 2*i+1, 2*i+2
        if l < n and arr[l] > arr[largest]: largest = l
        if r < n and arr[r] > arr[largest]: largest = r
        if largest != i:
            arr[i], arr[largest] = arr[largest], arr[i]
            heapify(arr, n, largest)
    for i in range(n//2-1, -1, -1): heapify(arr, n, i)
    for i in range(n-1, 0, -1):
        arr[0], arr[i] = arr[i], arr[0]; heapify(arr, i, 0)
    return arr

# ─── COUNTING / RADIX SORT ───────────────────────────────────────
def counting_sort(arr, k=None):
    if k is None: k = max(arr)+1
    count = [0]*k
    for v in arr: count[v] += 1
    for i in range(1, k): count[i] += count[i-1]
    output = [0]*len(arr)
    for v in reversed(arr): count[v]-=1; output[count[v]] = v
    return output

print(counting_sort([4,2,2,8,3,3,1]))  # [1,2,2,3,3,4,8]

a = [3,1,4,1,5,9,2,6]; quick_sort(a)
print("QuickSort:", a)  # [1,1,2,3,4,5,6,9]

---
## 16. Dynamic Programming Patterns <a id='16'></a>

### DP Framework
1. **Define state** — what does `dp[i]` represent?
2. **Transition** — how to compute `dp[i]` from smaller subproblems?
3. **Base case** — smallest valid inputs
4. **Answer** — which cell / combination gives the answer?

### Classic Patterns
| Pattern | Example | State |
|---|---|---|
| Linear DP | Fibonacci, Climbing Stairs | dp[i] |
| Knapsack | 0/1 Knapsack, Subset Sum | dp[i][w] |
| Unbounded Knapsack | Coin Change | dp[w] |
| LCS/LIS | Longest Common Subsequence | dp[i][j] |
| Interval DP | Matrix Chain, Burst Balloons | dp[i][j] |
| Tree DP | House Robber III | dp[node] |
| Bitmask DP | Traveling Salesman | dp[mask][i] |
| Digit DP | Count numbers with property | dp[pos][tight][...] |

In [ ]:
# ─── KNAPSACK VARIANTS ───────────────────────────────────────────

def knapsack_01(weights, values, W):  # 0/1 Knapsack O(nW)
    n = len(weights)
    dp = [0] * (W+1)
    for i in range(n):
        for w in range(W, weights[i]-1, -1):  # reverse!
            dp[w] = max(dp[w], dp[w-weights[i]] + values[i])
    return dp[W]

print(knapsack_01([1,3,4,5],[1,4,5,7], 7))  # 9

def coin_change(coins, amount):  # Unbounded Knapsack — min coins
    dp = [float('inf')] * (amount+1); dp[0] = 0
    for a in range(1, amount+1):
        for c in coins:
            if c <= a: dp[a] = min(dp[a], dp[a-c]+1)
    return dp[amount] if dp[amount] != float('inf') else -1

print(coin_change([1,5,6,9], 11))  # 2 (5+6)

def coin_change_ways(coins, amount):  # Count combinations
    dp = [0]*(amount+1); dp[0] = 1
    for c in coins:
        for a in range(c, amount+1): dp[a] += dp[a-c]
    return dp[amount]

print(coin_change_ways([1,2,5], 5))  # 4

# ─── LONGEST COMMON SUBSEQUENCE ──────────────────────────────────
def lcs(s, t):
    m, n = len(s), len(t)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(1,m+1):
        for j in range(1,n+1):
            if s[i-1]==t[j-1]: dp[i][j] = dp[i-1][j-1]+1
            else: dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    return dp[m][n]

print(lcs("abcde","ace"))  # 3

# ─── LONGEST INCREASING SUBSEQUENCE — O(n log n) ─────────────────
def lis(nums):
    tails = []  # tails[i] = smallest tail for LIS of length i+1
    for n in nums:
        pos = bisect.bisect_left(tails, n)
        if pos == len(tails): tails.append(n)
        else: tails[pos] = n
    return len(tails)

print(lis([10,9,2,5,3,7,101,18]))  # 4 (2,3,7,101)

# ─── EDIT DISTANCE ───────────────────────────────────────────────
def edit_distance(word1, word2):
    m, n = len(word1), len(word2)
    dp = list(range(n+1))
    for i in range(1,m+1):
        prev = dp[0]; dp[0] = i
        for j in range(1,n+1):
            tmp = dp[j]
            if word1[i-1] == word2[j-1]: dp[j] = prev
            else: dp[j] = 1 + min(prev, dp[j], dp[j-1])
            prev = tmp
    return dp[n]

print(edit_distance("horse", "ros"))  # 3

In [ ]:
# ─── INTERVAL DP ─────────────────────────────────────────────────

def burst_balloons(nums):  # O(n³)
    nums = [1] + nums + [1]; n = len(nums)
    dp = [[0]*n for _ in range(n)]
    for length in range(2, n):
        for l in range(0, n-length):
            r = l + length
            for k in range(l+1, r):
                dp[l][r] = max(dp[l][r],
                    dp[l][k] + nums[l]*nums[k]*nums[r] + dp[k][r])
    return dp[0][n-1]

print(burst_balloons([3,1,5,8]))  # 167

def matrix_chain(dims):  # Minimum mult cost O(n³)
    n = len(dims)-1
    dp = [[0]*n for _ in range(n)]
    for length in range(2, n+1):
        for i in range(n-length+1):
            j = i+length-1; dp[i][j] = float('inf')
            for k in range(i, j):
                dp[i][j] = min(dp[i][j],
                    dp[i][k]+dp[k+1][j]+dims[i]*dims[k+1]*dims[j+1])
    return dp[0][n-1]

print(matrix_chain([10,30,5,60]))  # 4500

# ─── BITMASK DP — Traveling Salesman Problem ─────────────────────
def tsp(dist):
    n = len(dist); FULL = (1<<n)-1
    dp = [[float('inf')]*n for _ in range(1<<n)]
    dp[1][0] = 0  # start at node 0, visited = {0}
    for mask in range(1<<n):
        for u in range(n):
            if not (mask>>u&1) or dp[mask][u] == float('inf'): continue
            for v in range(n):
                if mask>>v&1: continue
                new_mask = mask|(1<<v)
                dp[new_mask][v] = min(dp[new_mask][v], dp[mask][u]+dist[u][v])
    return min(dp[FULL][i]+dist[i][0] for i in range(1,n))

dist = [[0,10,15,20],[10,0,35,25],[15,35,0,30],[20,25,30,0]]
print("TSP minimum:", tsp(dist))  # 80

In [ ]:
# ─── PARTITION DP ────────────────────────────────────────────────

def can_partition(nums):  # Equal subset sum — O(n * sum)
    total = sum(nums)
    if total % 2: return False
    target = total // 2
    dp = {0}
    for n in nums:
        dp = {x+n for x in dp} | dp
        if target in dp: return True
    return False

print(can_partition([1,5,11,5]))  # True
print(can_partition([1,2,3,5]))   # False

def num_ways_target_sum(nums, target):  # Count subsets with given sum
    dp = defaultdict(int); dp[0] = 1
    for n in nums:
        new_dp = defaultdict(int)
        for s, cnt in dp.items():
            new_dp[s+n] += cnt; new_dp[s-n] += cnt
        dp = new_dp
    return dp[target]

print(num_ways_target_sum([1,1,1,1,1], 3))  # 5

# ─── PALINDROME PARTITIONING ─────────────────────────────────────
def min_cut_palindrome(s):
    n = len(s)
    # is_pal[i][j] = True if s[i:j+1] is palindrome
    is_pal = [[False]*n for _ in range(n)]
    for i in range(n-1,-1,-1):
        for j in range(i,n):
            is_pal[i][j] = s[i]==s[j] and (j-i<=2 or is_pal[i+1][j-1])
    dp = list(range(-1,n-1))  # dp[i] = min cuts for s[0:i+1]
    for i in range(1,n):
        if is_pal[0][i]: dp[i] = 0; continue
        for j in range(1,i+1):
            if is_pal[j][i]: dp[i] = min(dp[i], dp[j-1]+1)
    return dp[n-1]

print(min_cut_palindrome("aab"))     # 1 ("a" | "ab"? No, "a"|"a"|"b" = 2, but min is "aa"|"b" = 1)
print(min_cut_palindrome("ababbbabbababa"))  # 3

---
## 17. Greedy Algorithms <a id='17'></a>

### When Greedy Works
- Problem has **optimal substructure** + **greedy choice property**
- Prove by exchange argument: swapping greedy choice with any other doesn't improve solution

In [ ]:
# ─── INTERVAL PROBLEMS ───────────────────────────────────────────

def merge_intervals(intervals):
    intervals.sort(); merged = [intervals[0]]
    for s, e in intervals[1:]:
        if s <= merged[-1][1]: merged[-1][1] = max(merged[-1][1], e)
        else: merged.append([s, e])
    return merged

print(merge_intervals([[1,3],[2,6],[8,10],[15,18]]))  # [[1,6],[8,10],[15,18]]

def non_overlapping_intervals(intervals):  # Min removals to make non-overlapping
    intervals.sort(key=lambda x: x[1])  # sort by end time!
    end = float('-inf'); count = 0
    for s, e in intervals:
        if s >= end: end = e
        else: count += 1
    return count

print(non_overlapping_intervals([[1,2],[2,3],[3,4],[1,3]]))  # 1

def minimum_platforms(arrivals, departures):
    arrivals.sort(); departures.sort()
    platforms = result = 0; i = j = 0
    while i < len(arrivals):
        if arrivals[i] <= departures[j]: platforms+=1; i+=1
        else: platforms-=1; j+=1
        result = max(result, platforms)
    return result

print(minimum_platforms([9,9,10],[10,10,11]))  # 2

# ─── JUMP GAME II ────────────────────────────────────────────────
def jump_game_min_jumps(nums):
    jumps = cur_end = farthest = 0
    for i in range(len(nums)-1):
        farthest = max(farthest, i + nums[i])
        if i == cur_end: jumps+=1; cur_end = farthest
    return jumps

print(jump_game_min_jumps([2,3,1,1,4]))  # 2

# ─── ACTIVITY SELECTION ──────────────────────────────────────────
def max_activities(activities):  # activities = [(start, end)]
    activities.sort(key=lambda x: x[1])
    count = 1; last_end = activities[0][1]
    for s, e in activities[1:]:
        if s >= last_end: count+=1; last_end=e
    return count

print(max_activities([(1,2),(3,4),(0,6),(5,7),(8,9),(5,9)]))  # 4

# ─── GAS STATION ─────────────────────────────────────────────────
def can_complete_circuit(gas, cost):
    if sum(gas) < sum(cost): return -1
    start = tank = 0
    for i in range(len(gas)):
        tank += gas[i] - cost[i]
        if tank < 0: start = i+1; tank = 0
    return start

print(can_complete_circuit([1,2,3,4,5],[3,4,5,1,2]))  # 3

---
## 18. Bit Manipulation <a id='18'></a>

### Essential Operations
```python
x & (x-1)    # Clear lowest set bit
x & (-x)     # Isolate lowest set bit
x | (1<<k)   # Set bit k
x & ~(1<<k)  # Clear bit k
x ^ (1<<k)   # Toggle bit k
(x>>k) & 1   # Check bit k
bin(x).count('1')  # Popcount
```

In [ ]:
# ─── BIT TRICKS ──────────────────────────────────────────────────

def count_bits(n):    # Hamming weight
    count = 0
    while n: n &= n-1; count+=1
    return count

def is_power_of_two(n): return n > 0 and (n & n-1) == 0

def single_number(nums):  # One element appears once, rest twice
    return functools.reduce(lambda a,b: a^b, nums)

def single_number_iii(nums):  # Two elements appear once, rest twice
    xor = functools.reduce(lambda a,b: a^b, nums)
    diff_bit = xor & (-xor)   # rightmost set bit differs between the two
    a = 0
    for n in nums:
        if n & diff_bit: a ^= n
    return [a, xor^a]

print(single_number([2,2,1]))                # 1
print(single_number_iii([1,2,1,3,2,5]))      # [3,5] or [5,3]
print([count_bits(i) for i in range(9)])     # [0,1,1,2,1,2,2,3,1]
print(is_power_of_two(16), is_power_of_two(15))  # True False

# DP with bits: count bits for all numbers 0..n
def count_bits_dp(n):
    dp = [0]*(n+1)
    for i in range(1, n+1):
        dp[i] = dp[i>>1] + (i&1)  # dp[i] = dp[i/2] + last_bit
    return dp

print(count_bits_dp(8))  # [0,1,1,2,1,2,2,3,1]

# Subset enumeration with bitmask
def all_subsets_bitmask(nums):
    n = len(nums); result = []
    for mask in range(1<<n):
        subset = [nums[i] for i in range(n) if mask>>i&1]
        result.append(subset)
    return result

print(all_subsets_bitmask([1,2,3])[:4])  # [[], [1], [2], [1,2]]

# Enumerate all sub-masks of a mask (for bitmask DP)
def enumerate_submasks(mask):
    sub = mask
    while sub:
        yield sub
        sub = (sub-1) & mask

print(list(enumerate_submasks(0b1010)))  # [10, 8, 2] = [1010, 1000, 0010]

---
## 19. Math & Number Theory <a id='19'></a>

In [ ]:
# ─── PRIMES ──────────────────────────────────────────────────────

def sieve_of_eratosthenes(n):  # All primes up to n — O(n log log n)
    is_prime = [True]*(n+1); is_prime[0]=is_prime[1]=False
    for i in range(2, int(n**0.5)+1):
        if is_prime[i]:
            for j in range(i*i, n+1, i): is_prime[j] = False
    return [i for i in range(n+1) if is_prime[i]]

print(sieve_of_eratosthenes(30))  # [2,3,5,7,11,13,17,19,23,29]

def prime_factorization(n):   # O(√n)
    factors = []
    d = 2
    while d*d <= n:
        while n%d==0: factors.append(d); n//=d
        d+=1
    if n>1: factors.append(n)
    return factors

print(prime_factorization(360))  # [2,2,2,3,3,5]

# ─── GCD / LCM ───────────────────────────────────────────────────
def gcd(a, b): return a if b==0 else gcd(b, a%b)
def lcm(a, b): return a*b//gcd(a,b)
print(gcd(48, 18), lcm(4, 6))  # 6, 12

# ─── MODULAR ARITHMETIC ──────────────────────────────────────────
MOD = 10**9+7
def mod_pow(base, exp, mod): return pow(base, exp, mod)  # built-in fast
def mod_inv(a, mod): return pow(a, mod-2, mod)          # Fermat's little theorem

# Precompute factorials for combinations
def precompute_factorials(n, mod=10**9+7):
    fact = [1]*(n+1); inv_fact = [1]*(n+1)
    for i in range(1, n+1): fact[i] = fact[i-1]*i%mod
    inv_fact[n] = pow(fact[n], mod-2, mod)
    for i in range(n-1,-1,-1): inv_fact[i] = inv_fact[i+1]*(i+1)%mod
    def C(n,r): return fact[n]*inv_fact[r]%mod*inv_fact[n-r]%mod if 0<=r<=n else 0
    return fact, inv_fact, C

_, _, C = precompute_factorials(100)
print(C(10, 3))   # 120

# ─── EXTENDED GCD (Bezout) ───────────────────────────────────────
def extended_gcd(a, b):
    if b==0: return a, 1, 0
    g, x, y = extended_gcd(b, a%b)
    return g, y, x-(a//b)*y

g, x, y = extended_gcd(35, 15)
print(f"gcd={g}, Bezout: 35*{x} + 15*{y} = {35*x+15*y}")  # gcd=5

# ─── MATRIX EXPONENTIATION — nth Fibonacci in O(log n) ──────────
def mat_mul(A, B, mod=10**9+7):
    n = len(A)
    C = [[0]*n for _ in range(n)]
    for i in range(n):
        for k in range(n):
            for j in range(n):
                C[i][j] = (C[i][j]+A[i][k]*B[k][j])%mod
    return C

def mat_pow(M, n, mod=10**9+7):
    size = len(M)
    result = [[1 if i==j else 0 for j in range(size)] for i in range(size)]  # identity
    while n:
        if n&1: result = mat_mul(result, M, mod)
        M = mat_mul(M, M, mod); n>>=1
    return result

def fibonacci_fast(n):
    if n<=1: return n
    M = [[1,1],[1,0]]
    return mat_pow(M, n)[0][1]

print([fibonacci_fast(i) for i in range(10)])  # [0,1,1,2,3,5,8,13,21,34]

---
## 20. Advanced Patterns Quick Reference <a id='20'></a>

### Pattern Recognition Guide
| Clue in Problem | Pattern to Try |
|---|---|
| "Top K" | Heap / QuickSelect |
| "All permutations/subsets" | Backtracking |
| Sorted array + pair search | Two Pointers |
| Substring / subarray condition | Sliding Window |
| Tree traversal + accumulate | DFS with return value |
| DAG + ordering | Topological Sort |
| Grid shortest path | BFS |
| Overlapping subproblems | DP |
| Range query + update | Segment Tree / BIT |
| Dynamic connectivity | DSU |
| Maximize XOR | Binary Trie |
| Optimal local choice | Greedy |
| k-sorted streams | Min-Heap |
| "Next greater/smaller" | Monotonic Stack |

In [ ]:
# ─── QUICKSELECT — Kth smallest O(n) avg ─────────────────────────
def quickselect(nums, k):
    """Kth smallest (1-indexed). Average O(n), worst O(n²)."""
    import random
    def partition(lo, hi):
        pivot_idx = random.randint(lo, hi)
        nums[pivot_idx], nums[hi] = nums[hi], nums[pivot_idx]
        pivot = nums[hi]; i = lo
        for j in range(lo, hi):
            if nums[j] <= pivot: nums[i],nums[j]=nums[j],nums[i]; i+=1
        nums[i],nums[hi]=nums[hi],nums[i]; return i
    lo, hi = 0, len(nums)-1
    while lo < hi:
        p = partition(lo, hi)
        if p == k-1: break
        elif p < k-1: lo = p+1
        else: hi = p-1
    return nums[k-1]

print(quickselect([3,2,1,5,6,4], 2))  # 2 (2nd smallest)

# ─── SPARSE TABLE — Range Min/Max in O(1) ────────────────────────
class SparseTable:
    """Static RMQ: O(n log n) build, O(1) query."""
    def __init__(self, arr):
        n = len(arr); LOG = int(math.log2(n))+1 if n else 1
        self.table = [[float('inf')]*n for _ in range(LOG)]
        self.table[0] = arr[:]
        for j in range(1, LOG):
            for i in range(n-(1<<j)+1):
                self.table[j][i] = min(self.table[j-1][i],
                                       self.table[j-1][i+(1<<(j-1))])
        self.log2 = [0]*(n+1)
        for i in range(2, n+1): self.log2[i] = self.log2[i//2]+1

    def query(self, l, r):  # inclusive [l, r]
        j = self.log2[r-l+1]
        return min(self.table[j][l], self.table[j][r-(1<<j)+1])

sp = SparseTable([1,3,2,7,5,4,6])
print(sp.query(1,5))   # 2
print(sp.query(0,6))   # 1

# ─── MONOTONIC QUEUE — O(n) problems ────────────────────────────
def shortest_subarray_with_sum_k(nums, k):  # Sum >= k
    pre = [0]*(len(nums)+1)
    for i,v in enumerate(nums): pre[i+1] = pre[i]+v
    dq = deque(); res = float('inf')
    for i, v in enumerate(pre):
        while dq and v-pre[dq[0]] >= k:
            res = min(res, i-dq.popleft())
        while dq and pre[dq[-1]] >= v: dq.pop()
        dq.append(i)
    return res if res != float('inf') else -1

print(shortest_subarray_with_sum_k([2,-1,2], 3))  # 3

In [ ]:
# ─── HASHING TRICKS ──────────────────────────────────────────────

# Group anagrams
def group_anagrams(strs):
    groups = defaultdict(list)
    for s in strs: groups[tuple(sorted(s))].append(s)
    return list(groups.values())

print(group_anagrams(["eat","tea","tan","ate","nat","bat"]))

# Subarray sum equals K (count all)
def subarray_sum_k(nums, k):
    prefix_count = defaultdict(int); prefix_count[0] = 1
    cur = count = 0
    for n in nums:
        cur += n
        count += prefix_count[cur - k]
        prefix_count[cur] += 1
    return count

print(subarray_sum_k([1,1,1], 2))   # 2
print(subarray_sum_k([1,2,3], 3))   # 2

# ─── ROLLING HASH for DUPLICATE DETECTION ────────────────────────
def find_duplicate_substrings(s, length):
    BASE, MOD = 31, 10**9+9
    seen = set(); pw = pow(BASE, length, MOD); h = 0
    for i, c in enumerate(s):
        h = (h*BASE + ord(c)) % MOD
        if i >= length:
            h = (h - ord(s[i-length])*pw) % MOD
        if i >= length-1:
            if h in seen: return s[i-length+1:i+1]
            seen.add(h)
    return ""

print(find_duplicate_substrings("banana", 2))  # "an" or "na"

# Longest Duplicate Substring (binary search + rolling hash)
def longest_dup_substring(s):
    def has_dup(n):
        return bool(find_duplicate_substrings(s, n))
    lo, hi = 1, len(s)-1
    while lo < hi:
        mid = (lo+hi+1)//2
        if has_dup(mid): lo = mid
        else: hi = mid-1
    return find_duplicate_substrings(s, lo)

print(longest_dup_substring("banana"))   # "ana"
print(longest_dup_substring("abcd"))     # ""

In [ ]:
# ─── DIGIT DP ────────────────────────────────────────────────────

def count_digit_one(n):
    """Count total 1s in all numbers from 1 to n."""
    count = 0; factor = 1
    while factor <= n:
        lower = n % factor
        current = (n // factor) % 10
        higher = n // (factor * 10)
        if current == 0:   count += higher * factor
        elif current == 1: count += higher * factor + lower + 1
        else:              count += (higher + 1) * factor
        factor *= 10
    return count

print(count_digit_one(13))   # 6 (1,10,11,12,13)
print(count_digit_one(100))  # 21

# Generic digit DP template
@cache
def count_special(pos, tight, count, n_str):
    """Count numbers with equal 0s and 1s up to n_str."""
    if pos == len(n_str): return 1 if count == 0 else 0
    limit = int(n_str[pos]) if tight else 9
    total = 0
    for digit in range(0, limit+1):
        new_count = count + (1 if digit==1 else -1 if digit==0 else 0)
        if abs(new_count) > len(n_str)-pos: continue  # prune
        total += count_special(pos+1, tight and digit==limit, new_count, n_str)
    return total

n_str = "100"
print(f"Numbers with equal 0s and 1s up to 100: {count_special(0, True, 0, n_str)}")
count_special.cache_clear()

In [ ]:
# ─── COMPLEXITY CHEAT SHEET SUMMARY ─────────────────────────────
summary = """
╔══════════════════════════════════════════════════════════════════╗
║              COMPLEXITY QUICK REFERENCE                         ║
╠══════════════════════════════════════════════════════════════════╣
║ Array access         O(1)          Hash get/set  O(1) avg      ║
║ Binary search        O(log n)       Heap push/pop O(log n)      ║
║ Sorting              O(n log n)     Trie insert   O(m)          ║
║ BFS/DFS              O(V+E)         Dijkstra      O((V+E)logV)  ║
║ Merge sort           O(n log n)     Floyd-Warsh.  O(V³)         ║
║ Segment tree build   O(n)           Seg tree query O(log n)     ║
║ Fenwick update/query O(log n)       Kruskal MST   O(E log E)    ║
║ DSU find/union       O(α(n)) ≈ O(1) KMP search    O(n+m)        ║
║ DP knapsack          O(nW)          DP LCS        O(nm)         ║
║ Backtracking (perms) O(n!)          TSP bitmask   O(2ⁿ · n²)   ║
╚══════════════════════════════════════════════════════════════════╝
"""
print(summary)

print("🎉 DSA Cheat Sheet Complete! All cells ready to run and practice.")